# Week09 Capstone project - adaptive_beta by dimension

## Strategy
For Week 9, the policy uses **adaptive beta by dimension**.

The core idea is that higher-dimensional functions deserve slightly more exploration, so the UCB-style exploration coefficient grows with dimension:

$$\beta(d) = 0.12 + 0.06d$$

The notebook builds a **small, human-clean candidate pool** around the strongest anchors seen so far (`best`, `latest`, `second_best`) and scores them with:

$$\text{score}(x)=\mu(x)+\beta(d)\sigma(x)-0.18\,\text{dist}(x, best)-0.08\,\text{dist}(x, latest)+0.02\,\text{novelty}$$

This keeps the search local and reproducible while allowing a little more uncertainty-seeking behaviour as dimensionality increases.


In [1]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [6]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week09.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
56,8,1,2,0.630,0.630,NaN,NaN,NaN,NaN,NaN,NaN,1.979632
57,8,2,2,0.480,0.720,NaN,NaN,NaN,NaN,NaN,NaN,0.522171
58,8,3,3,0.380,0.580,0.480,NaN,NaN,NaN,NaN,NaN,-0.006062
59,8,4,4,0.397,0.394,0.360,0.441,NaN,NaN,NaN,NaN,0.202271
60,8,5,4,0.941,0.066,0.999,0.997,NaN,NaN,NaN,NaN,3671.957774
61,8,6,5,0.530,0.180,0.630,0.880,0.030,NaN,NaN,NaN,-0.475767
62,8,7,6,0.014,0.180,0.285,0.170,0.395,0.660,NaN,NaN,2.097019
63,8,8,8,0.033,0.436,0.010,0.322,0.990,0.044,0.096,0.701,9.587804


In [7]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d, rows

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [8]:
def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

cfg = {
    1: {'grid': 0.01,  'step': 0.01},
    2: {'grid': 0.01,  'step': 0.02},
    3: {'grid': 0.01,  'step': 0.01},
    4: {'grid': 0.001, 'step': 0.001},
    5: {'grid': 0.001, 'step': 0.001},
    6: {'grid': 0.01,  'step': 0.01},
    7: {'grid': 0.001, 'step': 0.001},
    8: {'grid': 0.001, 'step': 0.001},
}

def beta_by_dimension(d):
    return 0.12 + 0.06 * d

def build_candidates(best_x, latest_x, second_x, d, step, grid):
    cand = []
    anchors = [
        best_x,
        latest_x,
        second_x,
        (best_x + latest_x) / 2.0,
        (2*best_x + latest_x) / 3.0,
        (best_x + second_x) / 2.0,
    ]
    for a in anchors:
        cand.append(round_grid(a, grid))
    for i in range(d):
        for s in (-1.0, 1.0):
            x = best_x.copy()
            x[i] += s * step
            cand.append(round_grid(x, grid))
    for i in range(min(d, 4)):
        for j in range(i + 1, min(d, 4)):
            x = best_x.copy()
            x[i] += step
            x[j] -= step
            cand.append(round_grid(x, grid))
            x = best_x.copy()
            x[i] -= step
            x[j] += step
            cand.append(round_grid(x, grid))
    return np.unique(np.array(cand), axis=0)


In [9]:
def suggest_week9_point(function_id, history_df, seed=900):
    X, y, d, rows = load_combined(function_id, history_df)
    best_idx = int(np.argmax(y))
    second_idx = int(np.argsort(y)[-2])
    best_x = X[best_idx].copy()
    second_x = X[second_idx].copy()
    latest_x = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)[-1].copy()
    step = cfg[function_id]['step']
    grid = cfg[function_id]['grid']
    beta = beta_by_dimension(d)
    cand = build_candidates(best_x, latest_x, second_x, d, step, grid)
    dist = np.sqrt(((cand[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    keep = min_dist >= (0.0005 if grid == 0.001 else 0.005)
    cand = cand[keep]
    min_dist = min_dist[keep]
    gp = build_gp(seed + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)
    mean, std = gp.predict(cand, return_std=True)
    best_dist = np.sqrt(((cand - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((cand - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + beta * std - 0.18 * best_dist - 0.08 * latest_dist + 0.02 * min_dist
    idx = int(np.argmax(score))
    return {
        'function': function_id,
        'd': d,
        'x': cand[idx],
        'formatted': format_point(cand[idx]),
        'beta': beta,
        'step': step,
        'grid': grid,
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    r = suggest_week9_point(function_id, history)
    results.append(r)
    print(f"Function {function_id}: {r['formatted']}")
    print(f"  beta={r['beta']:.2f}, step={r['step']}, grid={r['grid']}")
    print(f"  best_anchor={r['best_anchor']}")
    print(f"  latest_anchor={r['latest_anchor']}")
    print()
